# ISeeSnow: Real topography — Voellmy — AVAC

This notebook runs one official ISeeSnow case without calibration, writes the required peak-flow-thickness and peak-flow-velocity rasters, and compares them on the supplied grid with participating-model submissions.


## Reproducible environment

The first code cell installs the repository's validation package and Python dependencies into the active kernel. A clean checkout also needs GNU Make and gfortran to compile the selected solver on first use. The pinned official ISeeSnow 1.0 dataset is downloaded automatically on first use. Before a case is run, the driver rebuilds AVAC once from this checkout so the recorded solver hash always represents the current source.


In [ ]:
from pathlib import Path
SEARCH_ROOT = Path.cwd().resolve()
REPOSITORY = next(candidate for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents) if (candidate / 'validation' / 'pyproject.toml').is_file())
%pip install -q -e {REPOSITORY / 'validation'}


In [ ]:
import os
from avac4qgis_validation import validation_case
case = validation_case('ISeeSnow', 'RealTopo')
CORES = max(1, os.cpu_count() or 1)
case.path


In [ ]:
from avac4qgis_validation.datasets import ensure_iseesnow
benchmark = ensure_iseesnow()
benchmark


## Prescribed benchmark configuration

The supplied real-terrain 5 m DEM and release polygon are run with $\mu=0.2$, $\xi=2000$ m/s², and 1.5 m normal release thickness. No peer-model result is used to select an AVAC parameter. The simulation ceiling is 1200 s and the native state is checked for practical arrest.


In [ ]:
CASE_NAME = 'RealTopo'
RESULTS_ROOT = Path(os.environ.get('AVAC_ISEESNOW_RESULTS_ROOT', case.path.parent)).expanduser().resolve()
case.run(case.path.parent / 'run_iseesnow_avac.py', '--case', CASE_NAME, '--workers', CORES, '--spatial-order', 2, '--results-root', RESULTS_ROOT, '--overwrite', cwd=case.path.parent)


## Run diagnostics and ISeeSnow submission

The summary records volume, duration, practical stop time, numerical controls, solver hash, and generated standard-format files.


In [ ]:
import json
summary = json.loads((RESULTS_ROOT / CASE_NAME / 'run_summary.json').read_text(encoding='utf-8'))
summary


## Direct peer comparison

Only peer fields with exactly matching dimensions, cell size, and cell-center coordinates are included; no shifting, clipping, padding, or resampling is performed.


In [ ]:
case.run(case.path.parent / 'compare_iseesnow.py', '--case', CASE_NAME, '--results-root', RESULTS_ROOT, '--output-root', RESULTS_ROOT, cwd=case.path.parent)
case.show(RESULTS_ROOT / 'plots' / f'{CASE_NAME}_pft_peer_comparison.png', RESULTS_ROOT / 'plots' / f'{CASE_NAME}_pfv_peer_comparison.png', RESULTS_ROOT / 'plots' / f'{CASE_NAME}_scalar_peer_comparison.png')
